# TP Bioinformatique COVID-19 - Partie 2
## Génome de référence et formats d'annotation

**Auteur : Marwa zidi**

**Durée estimée : 25 minutes**

### Objectifs
- Comprendre la structure du génome SARS-CoV-2
- Découvrir les formats GFF/GTF et GenBank
- Identifier les gènes et protéines du virus

---

In [ ]:
# Import des bibliothèques
import os
from Bio import SeqIO
from Bio.SeqFeature import SeqFeature, FeatureLocation
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

print("✅ Bibliothèques chargées")

## 1. Structure du génome SARS-CoV-2

Le SARS-CoV-2 possède un génome à **ARN simple brin positif** de ~30 kb.

### Organisation génomique

```
5' - [ORF1ab] - [S] - [ORF3a] - [E] - [M] - [ORF6-10] - [N] - 3'
```

**Gènes principaux :**
- **ORF1ab** : Polyprotéines (2/3 du génome)
- **S (Spike)** : Protéine de surface (cible des vaccins)
- **E (Envelope)** : Enveloppe virale
- **M (Membrane)** : Protéine membranaire
- **N (Nucleocapsid)** : Encapsidation de l'ARN

### Référence
**NC_045512.2** - Severe acute respiratory syndrome coronavirus 2 isolate Wuhan-Hu-1

In [ ]:
# Chargement du génome au format GenBank (avec annotations)
genbank_file = "/opt/covid_data/NC_045512.2.gb"

if os.path.exists(genbank_file):
    genome = SeqIO.read(genbank_file, "genbank")
    
    print(f"🦠 Génome : {genome.id}")
    print(f"📏 Longueur : {len(genome.seq):,} pb")
    print(f"📝 Description : {genome.description}")
    print(f"\n🔬 Nombre d'annotations : {len(genome.features)}")
else:
    print("⚠️ Fichier GenBank non trouvé. Utilisation du FASTA uniquement.")
    genome = SeqIO.read("/opt/covid_data/NC_045512.2.fasta", "fasta")

## 2. Exploration des annotations (format GenBank)

Le format **GenBank (.gb)** contient :
- La séquence nucléotidique
- Les **features** (gènes, CDS, régions)
- Les métadonnées (organisme, références, etc.)

**Structure d'une feature :**
```
Type: gene, CDS, mat_peptide
Location: 21563..25384
Qualifiers: /gene="S", /product="spike protein"
```

In [ ]:
# Extraction des gènes principaux
if os.path.exists(genbank_file):
    genes_info = []
    
    for feature in genome.features:
        if feature.type == "gene":
            gene_name = feature.qualifiers.get('gene', ['Unknown'])[0]
            start = int(feature.location.start)
            end = int(feature.location.end)
            length = end - start
            
            genes_info.append({
                'Gene': gene_name,
                'Start': start,
                'End': end,
                'Length (bp)': length,
                'Length (%)': f"{(length/len(genome.seq)*100):.1f}%"
            })
    
    # Affichage sous forme de tableau
    df_genes = pd.DataFrame(genes_info)
    print("📊 Gènes du SARS-CoV-2 :\n")
    print(df_genes.to_string(index=False))
else:
    print("⚠️ Impossible d'extraire les annotations sans fichier GenBank")

## 3. Visualisation du génome

Créons une représentation graphique du génome SARS-CoV-2 avec ses gènes.

In [ ]:
# Visualisation de l'organisation génomique
if os.path.exists(genbank_file):
    fig, ax = plt.subplots(figsize=(16, 4))
    
    # Ligne du génome
    ax.plot([0, len(genome.seq)], [0, 0], 'k-', linewidth=3)
    
    # Couleurs pour les différents gènes
    colors = {
        'ORF1ab': '#FF6B6B',
        'S': '#4ECDC4',
        'ORF3a': '#FFE66D',
        'E': '#95E1D3',
        'M': '#F38181',
        'ORF6': '#AA96DA',
        'ORF7a': '#FCBAD3',
        'ORF7b': '#A8D8EA',
        'ORF8': '#FFDAB9',
        'N': '#C7CEEA',
        'ORF10': '#B4A7D6'
    }
    
    for feature in genome.features:
        if feature.type == "gene":
            gene_name = feature.qualifiers.get('gene', ['Unknown'])[0]
            start = int(feature.location.start)
            end = int(feature.location.end)
            
            color = colors.get(gene_name, '#CCCCCC')
            
            # Rectangle pour le gène
            rect = mpatches.Rectangle((start, -0.3), end-start, 0.6, 
                                      facecolor=color, edgecolor='black', linewidth=1)
            ax.add_patch(rect)
            
            # Nom du gène
            ax.text((start+end)/2, 0.8, gene_name, 
                   ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    ax.set_xlim(0, len(genome.seq))
    ax.set_ylim(-1, 2)
    ax.set_xlabel('Position génomique (pb)', fontsize=12)
    ax.set_title('Organisation du génome SARS-CoV-2 (NC_045512.2)', fontsize=14, fontweight='bold')
    ax.set_yticks([])
    ax.spines['left'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Visualisation impossible sans fichier GenBank")

## 4. Focus sur le gène Spike (S)

Le gène **Spike** code pour la protéine de surface responsable de l'infection des cellules humaines via le récepteur ACE2.

**Importance :**
- Cible des anticorps neutralisants
- Cible des vaccins à ARNm
- Zone de mutations importantes (variants Alpha, Delta, Omicron)

In [ ]:
# Extraction de la séquence du gène Spike
if os.path.exists(genbank_file):
    for feature in genome.features:
        if feature.type == "gene" and feature.qualifiers.get('gene', [''])[0] == 'S':
            spike_start = int(feature.location.start)
            spike_end = int(feature.location.end)
            spike_seq = genome.seq[spike_start:spike_end]
            
            print(f"🎯 Gène Spike (S)")
            print(f"   Position : {spike_start:,} - {spike_end:,}")
            print(f"   Longueur : {len(spike_seq):,} pb")
            print(f"   Longueur protéine : {len(spike_seq)//3:,} acides aminés")
            print(f"\n   Séquence (50 premiers pb) :")
            print(f"   {spike_seq[:50]}")
            
            # Traduction en protéine
            spike_protein = spike_seq.translate()
            print(f"\n   Protéine (20 premiers AA) :")
            print(f"   {spike_protein[:20]}")
            
            break
else:
    print("⚠️ Fichier GenBank nécessaire pour cette analyse")

## 5. Format GFF/GTF (alternative à GenBank)

Les fichiers **GFF3** (General Feature Format) sont des fichiers tabulaires qui décrivent les annotations.

**Structure (9 colonnes) :**
```
seqid  source  type  start  end  score  strand  phase  attributes
```

**Exemple :**
```
NC_045512.2  RefSeq  gene  21563  25384  .  +  .  ID=gene-S;Name=S
```

In [ ]:
# Exemple de lecture d'un fichier GFF (si disponible)
gff_file = "/opt/covid_data/NC_045512.2.gff3"

if os.path.exists(gff_file):
    print("📄 Lecture du fichier GFF3 :\n")
    
    with open(gff_file, 'r') as f:
        lines = [line.strip() for line in f if not line.startswith('#')][:10]
        
    for line in lines:
        print(line)
else:
    print("ℹ️ Pas de fichier GFF disponible")
    print("\n💡 Le format GenBank (.gb) est plus complet et recommandé pour ce TP")

## 6. Exercice

**À faire :**
1. Identifier le gène le plus long
2. Calculer le pourcentage du génome codant pour des protéines
3. Extraire la séquence du gène N (nucléocapsid)

In [ ]:
# EXERCICE À COMPLÉTER

if os.path.exists(genbank_file):
    # 1. Gène le plus long
    max_length = 0
    longest_gene = ""
    total_coding = 0
    
    for feature in genome.features:
        if feature.type == "gene":
            gene_name = feature.qualifiers.get('gene', ['Unknown'])[0]
            length = len(feature)
            total_coding += length
            
            if length > max_length:
                max_length = length
                longest_gene = gene_name
    
    print(f"🏆 Gène le plus long : {longest_gene} ({max_length:,} pb)")
    
    # 2. Pourcentage codant
    percent_coding = (total_coding / len(genome.seq)) * 100
    print(f"📊 Pourcentage codant : {percent_coding:.2f}%")
    
    # 3. Séquence du gène N
    for feature in genome.features:
        if feature.type == "gene" and feature.qualifiers.get('gene', [''])[0] == 'N':
            n_seq = feature.extract(genome.seq)
            print(f"\n🧬 Gène N (Nucleocapsid) :")
            print(f"   Longueur : {len(n_seq)} pb")
            print(f"   Début : {n_seq[:30]}")
            break
else:
    print("⚠️ Exercice nécessite le fichier GenBank")

## 📝 Points clés

✅ Le génome SARS-CoV-2 fait ~30 kb (ARN simple brin +)  
✅ **Format GenBank** : Séquence + annotations complètes  
✅ **Format GFF/GTF** : Annotations tabulaires  
✅ **Gène Spike** : Cible thérapeutique majeure  
✅ **ORF1ab** : 2/3 du génome (polyprotéines)

---

**➡️ Passez au notebook suivant : `03_Controle_Qualite_Reads.ipynb`**